Données brut - sans features - désignation avec 4 fois plus de poids que description + extraction des unité de mesure + ajout de poids aux 3 premiers mot de designation
Méthode d'ajout de poids modifiée , plus de concaténation mais plutot directement dans le TFIDF

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import joblib
import re

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

# ============================================================
# FONCTIONS FEATURES (retournent TOUJOURS une Series)
# ============================================================
UNIT_PATTERN = r"(cm|mm|m|kg|g|mg|l|ml|cl|w|kw|v|mah|ah|hz|ghz|mhz|go|gb|to|tb|mp|px|fps|°c|°)"

def get_designation(X):
    return X["designation"].fillna("").astype(str)

def get_description(X):
    return X["description"].fillna("").astype(str)

def first_words_series(X, n=3):
    return (
        X["designation"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.split()
        .str[:n]
        .str.join(" ")
    )

def numbers_units_series(X):
    return (
        X["designation"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.findall(rf"\b\d+[.,]?\d*\s?{UNIT_PATTERN}\b")
        .str.join(" ")
    )

# ============================================================
# 1) Chargement données
# ============================================================
X_train = pd.read_csv(r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\projet_rakuten_2\Dernière_version\Préparation_des_données\X_train_non_nettoye_80.csv")
X_test  = pd.read_csv(r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\projet_rakuten_2\Dernière_version\Préparation_des_données\X_test_non_nettoye_20.csv")
Y       = pd.read_csv(r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\projet_rakuten_2\Dernière_version\Préparation_des_données\Y_train_encode.csv")

train = X_train.merge(Y, on="Unnamed: 0")
test  = X_test.merge(Y, on="Unnamed: 0")

y_train = train["prdtypecode_encoded"]
y_test  = test["prdtypecode_encoded"]

# ============================================================
# 2) TF-IDF (TES PARAMÈTRES)
# ============================================================
word_tfidf = TfidfVectorizer(
    max_features=120000,
    ngram_range=(1,2),
    strip_accents="unicode",
    lowercase=True,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2,5),
    min_df=2,
    lowercase=True
)

# ============================================================
# 3) ColumnTransformer
# ============================================================
features = ColumnTransformer([

    ("designation_word",
     Pipeline([
         ("select", FunctionTransformer(get_designation, validate=False)),
         ("tfidf", word_tfidf)
     ]),
     ["designation"]),

    ("designation_char",
     Pipeline([
         ("select", FunctionTransformer(get_designation, validate=False)),
         ("tfidf", char_tfidf)
     ]),
     ["designation"]),

    ("description_word",
     Pipeline([
         ("select", FunctionTransformer(get_description, validate=False)),
         ("tfidf", word_tfidf)
     ]),
     ["description"]),

    ("description_char",
     Pipeline([
         ("select", FunctionTransformer(get_description, validate=False)),
         ("tfidf", char_tfidf)
     ]),
     ["description"]),

    ("first_words",
     Pipeline([
         ("extract", FunctionTransformer(first_words_series, validate=False)),
         ("tfidf", TfidfVectorizer())
     ]),
     ["designation"]),

    ("numbers_units",
     Pipeline([
         ("extract", FunctionTransformer(numbers_units_series, validate=False)),
         ("tfidf", TfidfVectorizer())
     ]),
     ["designation"]),
])

# ============================================================
# 4) Pipeline final
# ============================================================
pipe = Pipeline([
    ("features", features),
    ("clf", LinearSVC(C=1.5, class_weight="balanced", max_iter=20000))
])

# ============================================================
# 5) Entraînement
# ============================================================
print("==> Entraînement...")
pipe.fit(train, y_train)

# ============================================================
# 6) Prédictions
# ============================================================
print("==> Prédictions...")
y_pred = pipe.predict(test)

print("F1 weighted :", f1_score(y_test, y_pred, average="weighted"))

# ============================================================
# 7) Sauvegarde
# ============================================================
joblib.dump(pipe, r"C:\Users\Mproo\Documents\Cours_DATASCIENTEST\projet_rakuten_2\Pipeline\modele_final_rakuten2.pkl")
print("✅ Modèle sauvegardé")


==> Entraînement...
==> Prédictions...
F1 weighted : 0.8622027778750723
✅ Modèle sauvegardé
